# 📐 Aula 18 — t de Student e qui-quadrado no controle de um serviço de pagamentos

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de um serviço de pagamentos online**. O tempo que o serviço leva para aprovar um pagamento (a **latência**) é normal, mas o time **não conhece o desvio padrão $\sigma$**: precisa estimá-lo pela amostra. A Aula 18 mostra o que muda quando isso acontece: para a **média**, a normal padrão dá lugar à **t de Student**; para a própria **variância**, entra a **qui-quadrado** ($\chi^2$).

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: as **tabelas** da normal (`norm`), da t (`t`) e da qui-quadrado (`chi2`) dentro do computador. Duas funções aparecem neste notebook:
  - `.ppf(p, gl)` dá o **valor crítico**: o número com área $p$ à **esquerda** dele, na distribuição com `gl` graus de liberdade (o que você lê na tabela);
  - `.pdf(x, gl)` dá a altura da curva, para desenhar o gráfico.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a tabela diz.

In [ ]:
import numpy as np                     # calculadora de números e sorteios
import matplotlib.pyplot as plt        # gráficos
from scipy.stats import norm, t, chi2  # as "tabelas"

rng = np.random.default_rng(42)        # sorteador, semente 42
print("Tudo pronto!")

## 1. Lendo a tabela t

### A situação
O time coletou a latência de **$n = 16$** pagamentos para estudar o tempo médio de aprovação, sem conhecer $\sigma$. Para os próximos cálculos, precisa do **valor crítico** $t_c$ com $P(T > t_c) = 0{,}05$.

### A pergunta
1. Quanto vale $t_c$? Como ele se compara ao valor da normal padrão ($z = 1{,}645$)?
2. E o valor com área **0,025 em cada cauda** (usado em intervalos de 95%), para vários tamanhos de amostra?

### A ferramenta da aula
Com $\sigma$ desconhecido, $T = \dfrac{\bar X - \mu}{S/\sqrt n}$ tem distribuição **t de Student** com $\nu = n - 1$ **graus de liberdade**. Aqui, $\nu = 15$.

### A resposta pela tabela
Na linha $\nu = 15$, coluna 0,05: $t_c = 1{,}753$ (maior que $z = 1{,}645$).

### O que o código faz
1. lê $t_c$ na "tabela" da t (área 0,95 à esquerda, ou seja, 0,05 à direita);
2. monta uma pequena tabela com o valor crítico de 0,025 na cauda para vários tamanhos de amostra, ao lado do valor da normal (1,96).

In [ ]:
n = 16
t_c = t.ppf(0.95, n - 1)                                       # área 0,05 à direita, ν = 15
print(f"ν = {n - 1}:  t_c = {t_c:.3f}   (normal padrão: z = {norm.ppf(0.95):.3f})\n")

print("  n    ν     t (0,025 na cauda)    z")
for n_ in (3, 5, 6, 16, 31, 101):
    print(f"{n_:4d} {n_ - 1:4d}        {t.ppf(0.975, n_ - 1):.3f}          {norm.ppf(0.975):.3f}")

**Como ler:**
- Com $\nu = 15$, o valor crítico é **1,753**, um pouco maior que o 1,645 da normal.
- Na tabela, quanto **menor** a amostra, **maior** o valor da t: com 3 observações ele é 4,303, mais que o dobro de 1,96. Com 101 observações já é 1,984, quase o valor da normal.

Por que a t "pede" valores maiores? É o que a próxima seção mostra.

## 2. Por que não dá para usar a normal quando $\sigma$ é estimado?

### A situação
Na verdade (só o notebook sabe), a latência é normal com $\mu = 120$ ms e $\sigma = 15$ ms. O time trabalha com amostras **pequenas**, de **$n = 5$** pagamentos.

### A pergunta
Se o time calcular $\dfrac{\bar X - \mu}{S/\sqrt n}$ (com o $S$ da amostra) e usar o limite **1,96** da normal, com que frequência o valor passa desse limite? Deveria ser **5%**.

### A ferramenta da aula
- Com $\sigma$ **conhecido**, $Z = \dfrac{\bar X - \mu}{\sigma/\sqrt n}$ é normal padrão: passa de 1,96 (em valor absoluto) em 5% das amostras;
- com $\sigma$ **estimado** por $S$, aparece uma incerteza a mais (o próprio $S$ varia de amostra para amostra), e $T$ segue a t com $\nu = 4$, cujas caudas são mais **pesadas**.

### O que o código faz
1. sorteia **100.000 amostras de 5 latências**;
2. calcula, em cada amostra, o $Z$ (com o $\sigma$ verdadeiro) e o $T$ (com o $S$ da amostra);
3. conta com que frequência cada um passa de 1,96, e com que frequência o $T$ passa do valor da tabela t (2,776 para $\nu = 4$);
4. desenha o histograma dos valores de $T$ com as curvas da normal e da t por cima.

In [ ]:
mu, sigma, n = 120, 15, 5
amostras = rng.normal(mu, sigma, size=(100_000, n))            # 100.000 amostras de 5 latências
xbarra = amostras.mean(axis=1)
S = amostras.std(axis=1, ddof=1)                               # desvio padrão amostral (divide por n - 1)

Z = (xbarra - mu) / (sigma / np.sqrt(n))                       # com σ conhecido
T = (xbarra - mu) / (S / np.sqrt(n))                           # com σ estimado por S
t_tab = t.ppf(0.975, n - 1)                                    # 2,776

print(f"|Z| > 1,96:   {(np.abs(Z) > 1.96).mean():.1%}   (esperado: 5%)")
print(f"|T| > 1,96:   {(np.abs(T) > 1.96).mean():.1%}   (usando o valor da normal: errado)")
print(f"|T| > {t_tab:.3f}: {(np.abs(T) > t_tab).mean():.1%}   (usando o valor da tabela t com ν = 4)")

x = np.linspace(-5, 5, 400)
plt.hist(T, bins=200, range=(-5, 5), density=True, color="lightsteelblue", edgecolor="none", label="valores de T simulados")
plt.plot(x, norm.pdf(x), color="black", linestyle="--", label="normal padrão")
plt.plot(x, t.pdf(x, n - 1), color="firebrick", linewidth=2, label="t com ν = 4")
plt.xlabel("valor da estatística")
plt.ylabel("Densidade")
plt.title("Com σ estimado, a estatística segue a t, não a normal")
plt.legend()
plt.show()

**Como ler:**
- O $Z$ (com $\sigma$ verdadeiro) passa de 1,96 em **5%** das amostras, como esperado.
- O $T$ (com $S$) passa de 1,96 em cerca de **12%** das amostras — mais que o dobro! Usar a normal com $\sigma$ estimado faria o time "ver problema" muito mais vezes do que deveria.
- Com o valor da **tabela t** (2,776), a frequência volta para **5%**.
- No gráfico, o histograma de $T$ segue a curva **vermelha** (t), que é mais baixa no centro e mais alta nas pontas que a normal: são as **caudas pesadas** da aula.

## 3. A qui-quadrado: a latência está variando demais?

### A situação
O contrato com os lojistas (o SLA) diz que o desvio padrão da latência é de **no máximo 10 ms** ($\sigma_0^2 = 100$). Numa amostra de **$n = 20$** pagamentos, a variância amostral deu **$S^2 = 150$ ms²**.

### A pergunta
Uma variância amostral de 150, quando a verdadeira seria 100, é um valor **muito improvável**? Ou pode ser só flutuação da amostra?

### A ferramenta da aula
Se a população é normal com variância $\sigma_0^2$, então
$$\chi^2 = \frac{(n-1)\,S^2}{\sigma_0^2}$$
tem distribuição **qui-quadrado** com $\nu = n - 1$ graus de liberdade. É o mesmo raciocínio do exemplo dos parafusos da aula.

### A resposta pela conta
- estatística: $\chi^2 = \dfrac{19 \times 150}{100} = 28{,}5$;
- valor crítico na tabela, $\nu = 19$, com área 0,05 à direita: $\chi^2_c = 30{,}144$.

Como $28{,}5 < 30{,}144$, o valor observado **não está** entre os 5% mais extremos.

### O que o código faz
1. calcula a estatística e o valor crítico da tabela;
2. calcula a chance de a qui-quadrado passar de 28,5;
3. **confere a teoria**: sorteia 100.000 amostras de 20 pagamentos de uma normal com $\sigma = 10$ (o SLA sendo cumprido), calcula $(n-1)S^2/\sigma_0^2$ em cada uma e vê com que frequência passa de 28,5;
4. desenha a curva da qui-quadrado com $\nu = 19$, a região dos 5% mais extremos e a posição do valor observado.

In [ ]:
n, S2, sigma0_2 = 20, 150, 100
estatistica = (n - 1) * S2 / sigma0_2
chi2_c = chi2.ppf(0.95, n - 1)                                 # área 0,05 à direita, ν = 19

amostras = rng.normal(0, 10, size=(100_000, n))                # SLA cumprido: σ = 10
simulada = (n - 1) * amostras.var(axis=1, ddof=1) / sigma0_2

print(f"Estatística: χ² = {estatistica:.2f}   valor crítico (ν = {n - 1}): {chi2_c:.3f}")
print(f"P(χ² > {estatistica})  pela tabela: {1 - chi2.cdf(estatistica, n - 1):.4f}   na simulação: {(simulada > estatistica).mean():.4f}")

x = np.linspace(0, 45, 400)
plt.plot(x, chi2.pdf(x, n - 1), color="steelblue", linewidth=2, label="qui-quadrado, ν = 19")
plt.fill_between(x, chi2.pdf(x, n - 1), where=(x > chi2_c), color="firebrick", label="5% mais extremos")
plt.axvline(estatistica, color="black", linestyle="--", label=f"valor observado = {estatistica}")
plt.xlabel("valor de (n - 1)S²/σ₀²")
plt.ylabel("Densidade")
plt.title("A variância observada é compatível com o SLA?")
plt.legend()
plt.show()

**Como ler:**
- O valor observado (28,5) fica **antes** da região vermelha, que começa em 30,144.
- Se o SLA estiver sendo cumprido, uma variância amostral tão grande quanto 150 (ou maior) aparece em cerca de **7,4%** das amostras de 20 pagamentos — pela tabela e pela simulação.
- A curva é **assimétrica** e só existe para valores positivos (é uma soma de quadrados), como a aula mostrou.

**Conclusão para o time:** com essa amostra, ainda não há evidência forte de que o SLA foi quebrado — mas o valor está perto do limite, e vale coletar mais dados. Decidir formalmente "quebrou ou não quebrou" é o assunto dos **testes de hipóteses**, no fim do curso.

## Para fechar

Neste notebook, o time de pagamentos usou as duas distribuições da Aula 18:

- **t de Student**, com $\nu = n - 1$: substitui a normal padrão quando $\sigma$ é **estimado** por $S$. Tem caudas mais pesadas, e por isso valores críticos maiores, sobretudo em amostras pequenas;
- **qui-quadrado**, com $\nu = n - 1$: é a distribuição de $(n-1)S^2/\sigma^2$, usada para estudar a **variância**.

Com a normal, a t e a qui-quadrado nas mãos, a **Aula 19** começa a construir **intervalos de confiança**: faixas de valores que, com uma confiança conhecida, contêm o parâmetro da população.